# 2 — Freeze 21 variants, create 480-row manifest, import 24 controls
Do not run any Stage 1 outcomes before this notebook completes. The complete Stage 0 bundle must be at `~/stage0`.


In [ ]:
import os, subprocess, json, csv
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; PY=Path.home()/"venv-stage1-ood/bin/python"
S0=Path.home()/"stage0"; OUT=Path.home()/"stage1"; OUT.mkdir(exist_ok=True)
required=[S0/"selected_high_delay.json",S0/"latency_calibration_episode_results.csv"]
missing=[str(x) for x in required if not x.exists()]
if missing: raise SystemExit("STOP: missing Stage 0 files: "+str(missing))
if json.loads((S0/"selected_high_delay.json").read_text())["high_added_delay_ms"]!=200: raise SystemExit("STOP: d* is not 200 ms")
assets=P/"libero/libero/assets"
if not assets.exists(): raise SystemExit("STOP: LIBERO-Plus assets are missing; complete notebook 1 asset installation")
env=os.environ.copy(); env.update({"PYTHONPATH":str(P),"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl"})
VAR=OUT/"stage1_resolved_variants.csv"; MAN=OUT/"stage1_manifest.csv"
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.resolve_stage1_variants","--output",str(VAR)],cwd=R,env=env,check=True)
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.make_stage1_manifest","--variants",str(VAR),"--selected-delay",str(S0/"selected_high_delay.json"),"--output",str(MAN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,env=env,check=True)
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.import_stage0_controls","--manifest",str(MAN),"--stage0-dir",str(S0),"--stage1-dir",str(OUT)],cwd=R,env=env,check=True)


In [ ]:
rows=list(csv.DictReader(open(OUT/"stage1_manifest.csv"))); results=list(csv.DictReader(open(OUT/"stage1_episode_results.csv")))
from collections import Counter
assert len(rows)==480 and len({r['run_id'] for r in rows})==480
assert Counter(r['scene_condition'] for r in rows)=={'ood':420,'id':60}
assert sum(r['reuse_stage0'].lower()=='true' for r in rows)==24 and len(results)==24
assert {r['seed'] for r in rows}=={'0','1','2','3','4'}
print("PASS manifest=480 OOD=420 ID=60 imported=24 new=456")
print("STOP HERE and paste this output plus the 21-row variant CSV for review before notebook 3.")
